In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# Одно смешивание вероятностей: 50% AGMFA + 50% старого ансамбля
Никакого нового обучения, inference, ранжировщика, подбора весов или температуры.
Старый ансамбль = 50% ResNet101(original) + 50% ResNet50(D4);
итоговые доли = 0.5 AGMFA + 0.25 ResNet101 + 0.25 ResNet50 D4.

Val использует **только train-only** модели. Здесь AGMFA — сохранённый B
best_loss эпохи22, cosine T_max24; val probabilities отдельного fresh final_train
с cosine22 в проекте отсутствуют. Это явно обозначенный proxy train-only recipe,
а не оценка train+val модели на её собственных обучающих объектах.
Train+val решения используются только в необязательном test export.


In [ ]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display

ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from archive.src.probability_fusion import analyze_fusion,save_validation_report,case_gallery

RUN_ID='probability_fusion_001'
RAW_DIR=REPOSITORY_ROOT / "data/raw"
AGMFA_VAL=ROOT/'artifacts/predictions/resnet50_agmfa_hard_target_shift_001/val_best_loss_probabilities.npz'
OLD_VAL=ROOT/'artifacts/predictions/quality_resnet101_saved_001/selected_probabilities.npz'
SHOW_GALLERIES=True
CASE_LIMIT=12
SMALL_OBJECT_IDS=[]  # val IDs из ручного анализа, если нужно


In [ ]:
agmfa_summary=json.loads((ROOT/'artifacts/logs/resnet50_agmfa_hard_target_shift_001/summary.json').read_text())
assert agmfa_summary['best_loss_epoch']==22
old_manifest=json.loads((ROOT/'artifacts/logs/quality_resnet101_saved_001/selection.json').read_text())
assert [(m['name'],m['tta'],m['weight']) for m in old_manifest['members']]==[('resnet101','none',.5),('stronger','d4',.5)]
assert all('final_train_val' not in m['checkpoint'] for m in old_manifest['members'])

result=analyze_fusion(AGMFA_VAL,OLD_VAL,RAW_DIR/'classes.csv',val_dir=RAW_DIR/'val')
context={'agmfa':'train-only B, best_loss epoch22, cosine T_max24; final_train cosine22 cache absent',
         'old_ensemble':old_manifest['recipe'],'selection':'weights fixed50/50; no search',
         'confidence_thresholds':[.9,.95]}
paths=save_validation_report(result,ROOT,RUN_ID,AGMFA_VAL,OLD_VAL,context)
display(result['metrics'])
display(result['complementarity'])
display(result['groups'])
display(result['per_class'].sort_values('old_ensemble_errors',ascending=False))


## Исправления и регрессии: отдельные объекты
NLL/Brier отражают качество вероятностей, Accuracy — argmax ответы.
Уверенная ошибка: максимум вероятностей >=90%/95% (пороги фиксированы заранее).
Улучшение NLL при потере одного верного ответа не доказывает улучшение test Accuracy.
Все изменения показаны открыто; 50/50 не подгоняется под эти таблицы.


In [ ]:
frame=result['frame']
changed=frame[frame.old_ensemble_correct != frame.blend_50_50_correct].copy()
columns=['id','true_label','agmfa_label','old_ensemble_label','blend_50_50_label',
         'agmfa_confidence','old_ensemble_confidence','blend_50_50_confidence',
         'brightness','contrast','detail']
class_names=result['classes'].set_index('label').class_name.to_dict()
view=changed[columns].copy()
for name in ['true_label','agmfa_label','old_ensemble_label','blend_50_50_label']:
    view[name]=view[name].map(class_names)
display(view)
if SHOW_GALLERIES:
    case_gallery(result,frame[~frame.old_ensemble_correct & frame.blend_50_50_correct],
                 'Blend исправил старый ансамбль',RAW_DIR/'val',CASE_LIMIT)
    case_gallery(result,frame[frame.old_ensemble_correct & ~frame.blend_50_50_correct],
                 'Blend испортил правильный ответ старого ансамбля',RAW_DIR/'val',CASE_LIMIT)


In [ ]:
if SHOW_GALLERIES:
    wrong=frame[~frame.blend_50_50_correct]
    case_gallery(result,wrong.nsmallest(CASE_LIMIT,'detail'),
                 'Blend ошибки: low-detail / blur proxy — проверить глазами',RAW_DIR/'val',CASE_LIMIT)
    case_gallery(result,wrong.nsmallest(CASE_LIMIT,'brightness'),
                 'Blend ошибки: самые тёмные сцены',RAW_DIR/'val',CASE_LIMIT)
    court_labels=result['classes'].loc[result['classes'].class_name.isin(['tennis_court','basketball_court']),'label']
    case_gallery(result,frame[frame.true_label.isin(court_labels) & (~frame.agmfa_correct | ~frame.old_ensemble_correct | ~frame.blend_50_50_correct)],
                 'Корты: проблемные объекты, все три предсказания',RAW_DIR/'val',CASE_LIMIT)
    if SMALL_OBJECT_IDS:
        case_gallery(result,frame[frame.id.isin(SMALL_OBJECT_IDS)],'Ручные small-object случаи',RAW_DIR/'val',CASE_LIMIT)
print('Detail/brightness — диагностические proxies, не разметка деградаций или размера объекта.')


## Вывод выполненной проверки
Старый ансамбль: **679/699**, смесь: **678/699**. Исправлено 3, испорчено 4.
NLL: **0.1323 → 0.0978**; ошибок с confidence ≥90%: **9 → 4**.
Один фиксированный test-кандидат сохранён благодаря улучшению вероятностей
и дополнению моделей; это **не доказанный выигрыш Accuracy**.
Подробный разбор: `../artifacts/reviews/probability_fusion_review.md`.
